# 🚗 Used Car Price Prediction — AdaBoost Regressor (Ensemble / Boosting)

**🇹🇷 Türkçe:**
Bu notebook, **CarDekho** ikinci el araç veri seti üzerinde bir aracın satış fiyatını (`selling_price`) tahmin eden bir **AdaBoost Regressor** modeli kuruyor. Veri seti 15.411 araca ait marka, model, yaş, kilometre, yakıt tipi, şanzıman, motor hacmi ve beygir gücü gibi bilgileri içeriyor.

**AdaBoost (Adaptive Boosting)** regresyonda da aynı mantıkla çalışır: zayıf öğreniciler (küçük karar ağaçları) **sırayla** eğitilir ve her yeni ağaç, bir öncekinin **en büyük hatayı yaptığı** örneklere daha fazla ağırlık vererek kurulur. Nihai tahmin, bu ağaçların ağırlıklı medyanıdır.

Bu notebook'un asıl teknik odağı **kategorik değişken kodlama stratejisidir**: veri setinde hem az kategorili (`fuel_type`: 5 değer) hem de **yüksek kardinaliteli** (`model`: 120 farklı değer) kolonlar bulunuyor ve bunlar için farklı encoding yöntemleri gerekiyor.

**🇬🇧 English:**
This notebook builds an **AdaBoost Regressor** on the **CarDekho** used-car dataset to predict a vehicle's selling price (`selling_price`). The dataset covers 15,411 cars with their brand, model, age, mileage, fuel type, transmission, engine size, and maximum power.

**AdaBoost (Adaptive Boosting)** works the same way for regression: weak learners (small decision trees) are trained **sequentially**, and each new tree is built with higher weights on the samples where the previous one made the **largest errors**. The final prediction is the weighted median of those trees.

The main technical focus of this notebook is the **categorical encoding strategy**: the dataset holds both low-cardinality columns (`fuel_type`: 5 values) and **high-cardinality** ones (`model`: 120 distinct values), and these call for different encoding methods.

## 📦 0. Kütüphaneler / Imports

**🇹🇷 Türkçe:**
Veri işleme, sayısal hesaplama ve görselleştirme için gereken temel kütüphaneler tek hücrede toplanıyor.

**🇬🇧 English:**
The core libraries for data handling, numeric computation, and visualization are collected in a single cell.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 📥 1. Veri Setini Yükleme / Load the Dataset

**🇹🇷 Türkçe:**
CarDekho ikinci el araç veri seti CSV dosyasından okunuyor. Hedef değişken `selling_price` — aracın Hint Rupisi cinsinden satış fiyatı.

**🇬🇧 English:**
The CarDekho used-car dataset is read from CSV. The target variable is `selling_price` — the car's sale price in Indian Rupees.

In [2]:
df = pd.read_csv("../../Data/17-cardekho.csv")

## 🔍 2. Veriye İlk Bakış / First Look at the Data

**🇹🇷 Türkçe:**
`head()`, `info()`, `describe()` ve `isnull()` ile veri setinin yapısı inceleniyor. Veri setinde eksik değer bulunmuyor. `info()` çıktısı 6 kolonun metin (string) tipinde olduğunu gösteriyor — bunlar modellemeden önce sayısala çevrilmesi gereken kategorik değişkenler.

`describe()` çıktısında dikkat çeken nokta hedef değişkenin **çok geniş bir aralığa** yayılmış olması: minimum 40.000, maksimum 39.500.000 (ortalamanın ~44 katı). Bu, veri setinde birkaç lüks araç (Ferrari, Rolls-Royce, Bentley) bulunmasından kaynaklanıyor ve regresyon skorlarını aşağı çeken temel etkenlerden biri.

**🇬🇧 English:**
The structure of the dataset is inspected with `head()`, `info()`, `describe()`, and `isnull()`. There are no missing values. The `info()` output shows 6 string columns — categorical variables that must be converted to numbers before modeling.

A notable point in `describe()` is how **widely spread** the target is: a minimum of 40,000 against a maximum of 39,500,000 (about 44× the mean). This comes from a handful of luxury cars in the data (Ferrari, Rolls-Royce, Bentley) and is one of the main factors pulling the regression scores down.

In [3]:
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  str    
 2   brand              15411 non-null  str    
 3   model              15411 non-null  str    
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  str    
 7   fuel_type          15411 non-null  str    
 8   transmission_type  15411 non-null  str    
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), str(6)
memory usage: 1.6 MB


In [5]:
df.describe()

,Unnamed: 0,vehicle_age,km_driven,mileage,engine,max_power,seats,selling_price
count,15411.000000,15411.000000,1.541100e+04,15411.000000,15411.000000,15411.000000,15411.000000,1.541100e+04
mean,9811.857699,6.036338,5.561648e+04,19.701151,1486.057751,100.588254,5.325482,7.749711e+05
std,5643.418542,3.013291,5.161855e+04,4.171265,521.106696,42.972979,0.807628,8.941284e+05
min,0.000000,0.000000,1.000000e+02,4.000000,793.000000,38.400000,0.000000,4.000000e+04
25%,4906.500000,4.000000,3.000000e+04,17.000000,1197.000000,74.000000,5.000000,3.850000e+05
50%,9872.000000,6.000000,5.000000e+04,19.670000,1248.000000,88.500000,5.000000,5.560000e+05
75%,14668.500000,8.000000,7.000000e+04,22.700000,1582.000000,117.300000,5.000000,8.250000e+05
max,19543.000000,29.000000,3.800000e+06,33.540000,6592.000000,626.000000,9.000000,3.950000e+07


In [6]:
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

## 🗑️ 3. Gereksiz Kolonun Atılması / Dropping the Index Column

**🇹🇷 Türkçe:**
`Unnamed: 0` kolonu, CSV'ye kaydedilirken yazılmış olan eski satır indeksinden ibaret — hiçbir bilgi taşımıyor. Modele girmesi halinde model, anlamsız bir sıra numarasından örüntü öğrenmeye çalışır; bu yüzden siliniyor.

**🇬🇧 English:**
The `Unnamed: 0` column is just the old row index written out when the CSV was saved — it carries no information. Leaving it in would let the model try to learn patterns from a meaningless row number, so it is dropped.

In [7]:
df = df.drop("Unnamed: 0", axis=1)

## 🔤 4. Kategorik Kolonların İncelenmesi / Inspecting the Categorical Columns

**🇹🇷 Türkçe:**
`select_dtypes` ile metin tipindeki kolonlar seçilip her birinin benzersiz değerleri listeleniyor. Ortaya çıkan tablo, encoding stratejisini doğrudan belirliyor:

| Kolon | Benzersiz değer sayısı |
|---|---|
| `car_name` | 121 |
| `model` | 120 |
| `brand` | 32 |
| `fuel_type` | 5 |
| `seller_type` | 3 |
| `transmission_type` | 2 |

**Kardinalite** (bir kategorik kolonun kaç farklı değer aldığı) burada kritik: 3 kategorili bir kolonu One-Hot Encoding ile açmak 3 yeni kolon yaratırken, 120 kategorili bir kolonu açmak 120 yeni kolon yaratır ve bu **boyut laneti (curse of dimensionality)** problemine yol açar.

**🇬🇧 English:**
`select_dtypes` selects the string columns and the unique values of each are listed. The resulting picture directly determines the encoding strategy:

| Column | Unique values |
|---|---|
| `car_name` | 121 |
| `model` | 120 |
| `brand` | 32 |
| `fuel_type` | 5 |
| `seller_type` | 3 |
| `transmission_type` | 2 |

**Cardinality** — how many distinct values a categorical column takes — is critical here: one-hot encoding a 3-category column creates 3 new columns, while one-hot encoding a 120-category column creates 120 and leads straight into the **curse of dimensionality**.

In [8]:
cat_cols = df.select_dtypes(include="object")

C:\Users\efeka\AppData\Local\Temp\ipykernel_22880\4070191198.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object")


In [9]:
cat_cols

,car_name,brand,model,seller_type,fuel_type,transmission_type
0,Maruti Alto,Maruti,Alto,Individual,Petrol,Manual
1,Hyundai Grand,Hyundai,Grand,Individual,Petrol,Manual
2,Hyundai i20,Hyundai,i20,Individual,Petrol,Manual
3,Maruti Alto,Maruti,Alto,Individual,Petrol,Manual
4,Ford Ecosport,Ford,Ecosport,Dealer,Diesel,Manual
...,...,...,...,...,...,...
15406,Hyundai i10,Hyundai,i10,Dealer,Petrol,Manual
15407,Maruti Ertiga,Maruti,Ertiga,Dealer,Petrol,Manual
15408,Skoda Rapid,Skoda,Rapid,Dealer,Diesel,Manual
15409,Mahindra XUV500,Mahindra,XUV500,Dealer,Diesel,Manual


In [10]:
for cols in cat_cols:
    print(f"{cols}: \n{df[cols].unique()} \n")

car_name: 
<StringArray>
[          'Maruti Alto',         'Hyundai Grand',           'Hyundai i20',
         'Ford Ecosport',        'Maruti Wagon R',           'Hyundai i10',
         'Hyundai Venue',          'Maruti Swift',         'Hyundai Verna',
        'Renault Duster',
 ...
      'Mahindra Alturas',           'Tata Altroz',              'Lexus NX',
          'Kia Carnival',        'Mercedes-AMG C',              'Lexus RX',
     'Rolls-Royce Ghost', 'Maserati Quattroporte',             'Isuzu MUX',
          'Force Gurkha']
Length: 121, dtype: str 

brand: 
<StringArray>
[       'Maruti',       'Hyundai',          'Ford',       'Renault',
          'Mini', 'Mercedes-Benz',        'Toyota',    'Volkswagen',
         'Honda',      'Mahindra',        'Datsun',          'Tata',
           'Kia',           'BMW',          'Audi',    'Land Rover',
        'Jaguar',            'MG',         'Isuzu',       'Porsche',
         'Skoda',         'Volvo',         'Lexus',          'Jeep',


## 0️⃣1️⃣ 5. İkili Kolonun Eşlenmesi / Mapping the Binary Column

**🇹🇷 Türkçe:**
`transmission_type` yalnızca iki değer alıyor (`Manual` / `Automatic`), bu yüzden One-Hot Encoding'e gerek yok — doğrudan `map()` ile 0 ve 1'e çevriliyor. İki kategorili bir kolonu one-hot ile açmak birbirinin tam tersi (ve dolayısıyla gereksiz) iki kolon üretirdi.

**🇬🇧 English:**
`transmission_type` takes only two values (`Manual` / `Automatic`), so one-hot encoding is unnecessary — it is mapped directly to 0 and 1 with `map()`. One-hot encoding a two-category column would produce two perfectly complementary (and therefore redundant) columns.

In [11]:
df["transmission_type"] = df["transmission_type"].map({"Manual":0,"Automatic":1})

In [12]:
df["transmission_type"]

0        0
1        0
2        0
3        0
4        0
        ..
15406    0
15407    0
15408    0
15409    0
15410    1
Name: transmission_type, Length: 15411, dtype: int64

## ♻️ 6. Yedekli Kolonun Atılması / Dropping the Redundant Column

**🇹🇷 Türkçe:**
`car_name` kolonu aslında `brand` ve `model` kolonlarının birleşiminden ibaret (`"Maruti Alto"` = `"Maruti"` + `"Alto"`). Üç kolonu birden tutmak aynı bilgiyi tekrarlamaktan başka bir işe yaramaz, üstelik 121 kategorili bir kolon daha eklemiş oluruz. Bu yüzden `car_name` siliniyor ve bilgi `brand` + `model` üzerinden korunuyor.

**🇬🇧 English:**
The `car_name` column is simply the concatenation of `brand` and `model` (`"Maruti Alto"` = `"Maruti"` + `"Alto"`). Keeping all three only repeats the same information while adding another 121-category column. So `car_name` is dropped and the information is preserved through `brand` + `model`.

In [13]:
df = df.drop("car_name", axis=1)

## 🎯 7. Encoding Stratejisi: One-Hot vs Target Encoding

**🇹🇷 Türkçe:**
Kolonlar kardinalitelerine göre iki gruba ayrılıyor:

**One-Hot Encoding** → `seller_type` (3), `fuel_type` (5)
Her kategori için 0/1 değerli ayrı bir kolon üretir. Kategoriler arasında **hiçbir sıra ilişkisi varsaymaz** — `Petrol`'ün `Diesel`'den "büyük" olduğu gibi yanlış bir sinyal vermez. Az kategorili kolonlarda ideal.

**Target Encoding** → `brand` (32), `model` (120)
Her kategoriyi, o kategorideki satırların **hedef değişken ortalamasıyla** değiştirir. Örneğin `"Ferrari"` kategorisi, tüm Ferrari'lerin ortalama satış fiyatıyla temsil edilir. Böylece 120 kategorili bir kolon **tek bir sayısal kolona** indirgenir, üstelik kategorinin fiyatla ilişkisi doğrudan kodlanmış olur.

⚠️ **Target Encoding'in riski — veri sızıntısı:** Hedef değişkeni kullanarak özellik ürettiğimiz için, kodlamanın tüm veri seti üzerinde hesaplanması test bilgisinin modele sızmasına yol açar. `sklearn`'ün `TargetEncoder`'ı bunu içeriden **çapraz doğrulamalı (cross-fitting)** hesaplayarak çözer: her satırın kodlaması, o satırı içermeyen fold'lardan hesaplanır.

**🇬🇧 English:**
The columns are split into two groups by cardinality:

**One-Hot Encoding** → `seller_type` (3), `fuel_type` (5)
Creates a separate 0/1 column per category. It assumes **no ordering** between categories — it never signals that `Petrol` is somehow "greater than" `Diesel`. Ideal for low-cardinality columns.

**Target Encoding** → `brand` (32), `model` (120)
Replaces each category with the **mean of the target** for the rows in that category. The `"Ferrari"` category, for instance, is represented by the average selling price of all Ferraris. A 120-category column collapses into **a single numeric column**, and the category's relationship with price is encoded directly.

⚠️ **The risk of target encoding — data leakage:** because the feature is built from the target, computing the encoding over the whole dataset leaks test information into the model. Scikit-learn's `TargetEncoder` solves this internally with **cross-fitting**: each row's encoding is computed from the folds that exclude that row.

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler

In [15]:
one_hot_cols = ["seller_type", "fuel_type"]
target_cols = ["brand","model"]

## 🏗️ 8. Ön İşleme Mimarisi / Building the Preprocessing Pipeline

**🇹🇷 Türkçe:**
Her encoder kendi `Pipeline`'ına, ikisi birden tek bir `ColumnTransformer`'a yerleştiriliyor.

**Burada anlaşılması gereken kritik davranış:** `ColumnTransformer` içindeki transformer'lar **paralel** çalışır — hiçbiri diğerinin çıktısını girdi olarak almaz, hepsi orijinal `X`'in kendi ham kolonlarına uygulanır. Bu yüzden `StandardScaler`'ı `ColumnTransformer`'a ayrı bir satır olarak eklemek işe yaramaz; scaler, target encoder'ın ürettiği sayıları değil ham `"Volvo"`, `"Maruti"` gibi metinleri görür ve hata verir. Doğru yöntem, scaler'ı **encoder'ın kendi pipeline'ının içine zincirlemektir** — burada yapılan da bu.

**`TargetEncoder(target_type="continuous")`:** `TargetEncoder`, hedefin tipini `y`'nin veri tipinden otomatik tahmin eder. `selling_price` tam sayı (`int64`) olduğu için sklearn bunu yanlışlıkla bir sınıflandırma hedefi sanıp binlerce farklı fiyatı ayrı birer "sınıf" olarak ele almaya çalışır. `target_type="continuous"` bunu açıkça regresyon olarak belirtir.

**`remainder="passthrough"`:** Listelenmemiş kolonlar (`vehicle_age`, `km_driven`, `mileage`, `engine`, `max_power`, `seats`, `transmission_type`) olduğu gibi geçirilir. Varsayılan değer `"drop"` olduğu için bu parametre **atlanırsa bu 7 sayısal kolon sessizce silinir** — hiçbir hata mesajı alınmadan.

**🇬🇧 English:**
Each encoder goes into its own `Pipeline`, and both are combined in a single `ColumnTransformer`.

**The critical behaviour to understand here:** transformers inside a `ColumnTransformer` run **in parallel** — none of them receives another's output; each is applied to its own raw columns of the original `X`. That is why adding `StandardScaler` as a separate entry in the `ColumnTransformer` does not work: the scaler would see raw strings like `"Volvo"` and `"Maruti"` rather than the numbers produced by the target encoder, and would fail. The correct approach is to **chain the scaler inside the encoder's own pipeline** — which is what is done here.

**`TargetEncoder(target_type="continuous")`:** `TargetEncoder` infers the target type from the dtype of `y`. Since `selling_price` is an integer (`int64`), scikit-learn mistakes it for a classification target and tries to treat thousands of distinct prices as separate "classes". `target_type="continuous"` states explicitly that this is regression.

**`remainder="passthrough"`:** columns not listed (`vehicle_age`, `km_driven`, `mileage`, `engine`, `max_power`, `seats`, `transmission_type`) are passed through unchanged. Since the default is `"drop"`, **omitting this parameter would silently delete those 7 numeric columns** — with no error raised.

In [16]:
target_enc_pipe = Pipeline([
    ("target_encoder", TargetEncoder(target_type="continuous")),
    ("scaler", StandardScaler()),
])
one_hot_enc_pipe = Pipeline([("one_hot_encoder", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([
    ("target_trans", target_enc_pipe, target_cols),
    ("one_hot_trans", one_hot_enc_pipe, one_hot_cols),
], remainder="passthrough")

## ✂️ 9. Özellik/Hedef Ayrımı ve Train-Test Split

**🇹🇷 Türkçe:**
Veri `X` (özellikler) ve `y` (hedef: `selling_price`) olarak ayrılıyor, ardından %80 eğitim / %20 test olarak bölünüyor. `random_state=42` ile bölme tekrarlanabilir hale getiriliyor.

**🇬🇧 English:**
The data is separated into `X` (features) and `y` (target: `selling_price`), then split 80% train / 20% test. `random_state=42` makes the split reproducible.

In [17]:
X = df.drop("selling_price", axis=1)
y = df["selling_price"]

In [18]:
from sklearn.model_selection import train_test_split

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## ⚙️ 10. Dönüşümün Uygulanması / Applying the Transformations

**🇹🇷 Türkçe:**
Ön işleme yalnızca eğitim setinde `fit_transform`, test setinde ise sadece `transform` ile uygulanıyor — bu, sızıntısız kurulumun temel kuralı.

`fit_transform(X_train, y_train)` çağrısında **`y_train`'in de verilmesi zorunlu**, çünkü `TargetEncoder` kategori ortalamalarını hesaplamak için hedef değişkene ihtiyaç duyar. `transform(X_test)` ise `y_test` almaz — test seti için eğitimde öğrenilen kodlamalar kullanılır, yani test hedefleri modele hiç gösterilmez.

**🇬🇧 English:**
Preprocessing is applied with `fit_transform` on the training set and `transform` only on the test set — the basic rule of a leakage-free setup.

Passing **`y_train` to `fit_transform(X_train, y_train)` is mandatory**, because `TargetEncoder` needs the target to compute the per-category means. `transform(X_test)` takes no `y_test` — the test set reuses the encodings learned during training, so the test targets are never shown to the model.

In [20]:
X_train_trans = preprocessor.fit_transform(X_train, y_train)
X_test_trans = preprocessor.transform(X_test)

## 🚀 11. AdaBoost Regressor Modelinin Kurulması / Training the Model

**🇹🇷 Türkçe:**
`AdaBoostRegressor` varsayılan parametrelerle kuruluyor: 50 tane `max_depth=3` karar ağacı, `learning_rate=1.0`, `loss="linear"`. Regresyon versiyonunda her turda örnek ağırlıkları, tahmin **hatasının büyüklüğüne** göre güncellenir — en çok yanılınan araçlar bir sonraki ağaçta daha fazla ağırlık kazanır.

**🇬🇧 English:**
`AdaBoostRegressor` is fit with its defaults: 50 `max_depth=3` decision trees, `learning_rate=1.0`, and `loss="linear"`. In the regression variant, sample weights are updated at each round according to the **magnitude of the prediction error** — the cars predicted worst gain more weight in the next tree.

In [21]:
from sklearn.ensemble import AdaBoostRegressor

In [22]:
ada_reg = AdaBoostRegressor()

In [23]:
ada_reg.fit(X_train_trans, y_train)

,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.If ``None``, then the base estimator is:class:`~sklearn.tree.DecisionTreeRegressor` initialized with`max_depth=3`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",50
,"learning_rate learning_rate: float, default=1.0Weight applied to each regressor at each boosting iteration. A higherlearning rate increases the contribution of each regressor. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",1.0
,"loss loss: {'linear', 'square', 'exponential'}, default='linear'The loss function to use when updating the weights after eachboosting iteration.",'linear'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.In addition, it controls the bootstrap of the weights used to train the`estimator` at each boosting iteration.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
Name,Type,Value
estimator_ estimator_: estimatorThe base estimator from which the ensemble is grown... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,DecisionTreeRegressor,DecisionTreeR...r(max_depth=3)
estimator_errors_ estimator_errors_: ndarray of floatsRegression error for each estimator in the boosted ensemble.,"ndarray[float64](50,)","[0.01,0.03,0.03,...,0.37,0.32,0.3 ]"
estimator_weights_ estimator_weights_: ndarray of floatsWeights for each estimator in the boosted ensemble.,"ndarray[float64](50,)","[5.06,3.32,3.58,...,0.53,0.75,0.85]"
estimators_ estimators_: list of regressorsThe collection of fitted sub-estimators.,list,"[DecisionTreeR...ate=455874341), DecisionTreeR...te=1520373922), DecisionTreeR...te=1152790737), DecisionTreeR...tate=19520265), ...]"
"feature_importances_ feature_importances_: ndarray of shape (n_features,)The impurity-based feature importances if supported by the``estimator`` (when based on decision trees).Warning: impurity-based feature importances can be misleading forhigh cardinality features (many unique values). See:func:`sklearn.inspection.permutation_importance` as an alternative.","ndarray[float64](17,)","[0.03,0.04,0. ,...,0.05,0.41,0. ]"


In [24]:
y_pred = ada_reg.predict(X_test_trans)

## 📈 12. Model Değerlendirmesi / Model Evaluation

**🇹🇷 Türkçe:**
Model `r2_score` ve `mean_squared_error` ile değerlendiriliyor.

**R² (belirlilik katsayısı)**, modelin hedef değişkendeki varyansın ne kadarını açıklayabildiğini gösterir: 1 mükemmel tahmin, 0 ise "her seferinde ortalamayı söylemek" kadar iyi demektir. Varsayılan parametrelerle elde edilen **R² ≈ 0.63**, fiyat varyansının yaklaşık üçte ikisinin açıklanabildiği anlamına geliyor. Bir sonraki bölümde bu skorun hiperparametre optimizasyonuyla ne kadar yükseltilebildiği görülecek.

MSE değerinin çok büyük görünmesi (10¹¹ mertebesinde) alarm verici değil — hata, hedef değişkenin biriminin **karesi** cinsinden ölçülür ve fiyatlar milyonlar mertebesindedir. Yorumlanabilir bir hata değeri için `mean_absolute_error` veya MSE'nin karekökü (RMSE) tercih edilebilir.

**🇬🇧 English:**
The model is evaluated with `r2_score` and `mean_squared_error`.

**R² (the coefficient of determination)** shows how much of the variance in the target the model can explain: 1 is a perfect fit, 0 is no better than always predicting the mean. The **R² ≈ 0.63** obtained with default parameters means roughly two-thirds of the price variance is explained. The next section shows how far tuning can push that score.

The very large MSE (on the order of 10¹¹) is not alarming — error is measured in the **square** of the target's unit, and prices run into the millions. For an interpretable error figure, `mean_absolute_error` or the square root of MSE (RMSE) is preferable.

In [25]:
from sklearn.metrics import mean_squared_error, r2_score

In [26]:
print(r2_score(y_test, y_pred))
print(mean_squared_error(y_test, y_pred))

0.633353872325159
276004528464.802


## 🎛️ 13. Hiperparametre Optimizasyonu / Hyperparameter Tuning with `GridSearchCV`

**🇹🇷 Türkçe:**
`GridSearchCV` ile 5 katlı çapraz doğrulama üzerinden en iyi parametre kombinasyonu aranıyor:

* **`n_estimators`** — kaç zayıf öğrenici sırayla eğitilecek.
* **`learning_rate`** — her öğrenicinin nihai tahmindeki ağırlığı. `n_estimators` ile **ters ilişkilidir**: öğrenme oranını düşürdükçe genellikle daha fazla ağaca ihtiyaç duyulur.
* **`loss`** — ağırlık güncellemesinde hatanın nasıl cezalandırılacağı. `"linear"` hatayla orantılı, `"square"` hatanın karesiyle, `"exponential"` ise üstel olarak cezalandırır. Bu veri setinde birkaç lüks araç uç fiyatlara sahip olduğu için, `"square"` ve `"exponential"` bu örneklere aşırı ağırlık verip modeli bozabilir — `"linear"`'ın kazanması beklenir, ama bunu varsaymak yerine ölçüyoruz.
* **`estimator`** — zayıf öğrenicinin kendisi; farklı ağaç derinlikleri deneniyor.

**🇬🇧 English:**
`GridSearchCV` searches for the best parameter combination with 5-fold cross-validation:

* **`n_estimators`** — how many weak learners are trained in sequence.
* **`learning_rate`** — the weight each learner carries in the final prediction. It is **inversely related** to `n_estimators`: the lower the learning rate, the more trees are usually needed.
* **`loss`** — how the error is penalized during the weight update. `"linear"` penalizes proportionally to the error, `"square"` by its square, and `"exponential"` exponentially. Since a few luxury cars carry extreme prices in this dataset, `"square"` and `"exponential"` may over-weight those samples and distort the model — `"linear"` is expected to win, but this is measured rather than assumed.
* **`estimator`** — the weak learner itself; several tree depths are tried.

In [27]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 0.5, 1],
    "loss": ["linear", "square", "exponential"],
    "estimator": [
        DecisionTreeRegressor(max_depth=3),
        DecisionTreeRegressor(max_depth=5),
    ],
}

grid_search = GridSearchCV(
    estimator=AdaBoostRegressor(random_state=42),
    param_grid=param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    verbose=2,
)

In [28]:
grid_search.fit(X_train_trans, y_train)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",AdaBoostRegre...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'estimator': [DecisionTreeR...r(max_depth=3), DecisionTreeR...r(max_depth=5)], 'learning_rate': [0.01, 0.1, ...], 'loss': ['linear', 'square', ...], 'n_estimators': [50, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more abou

In [29]:
print(grid_search.best_params_)
print(grid_search.best_score_)

{'estimator': DecisionTreeRegressor(max_depth=5), 'learning_rate': 0.1, 'loss': 'linear', 'n_estimators': 200}
0.8642991431502931


In [30]:
best_ada_reg = grid_search.best_estimator_
y_pred_best = best_ada_reg.predict(X_test_trans)

print(r2_score(y_test, y_pred_best))
print(mean_squared_error(y_test, y_pred_best))

0.9069107224597707
70075913021.52217


## ✅ 14. Sonuç / Conclusion

**🇹🇷 Türkçe:**
Bu notebook boyunca:

* Kategorik kolonlar **kardinalitelerine göre** ayrıştırıldı: az kategorili olanlar `OneHotEncoder`, yüksek kardinaliteli olanlar (`brand`, `model`) `TargetEncoder` ile kodlandı.
* `car_name`'in `brand` + `model` ile **yedekli** olduğu fark edilip kolon silindi.
* `ColumnTransformer`'ın transformer'ları **paralel** çalıştırdığı, dolayısıyla `StandardScaler`'ın encoder çıktısını değil ham kolonları göreceği somut bir hata üzerinden öğrenildi; scaler encoder'ın pipeline'ına zincirlendi.
* `TargetEncoder`'ın hedef tipini `y`'nin dtype'ından tahmin ettiği ve tam sayı bir fiyat kolonunu sınıflandırma hedefi sanabileceği görülüp `target_type="continuous"` ile açıkça belirtildi.
* Ön işleme yalnızca eğitim setinde `fit` edilerek **veri sızıntısı** engellendi.
* Varsayılan model **R² = 0.633** verdi; `GridSearchCV` ile `n_estimators`, `learning_rate`, `loss` ve ağaç derinliği birlikte optimize edilerek skor **R² = 0.907**'ye yükseltildi (MSE 2.76×10¹¹ → 7.01×10¹⁰, yani hata **dörtte bire** indi).
* Kazanan kombinasyon: `DecisionTreeRegressor(max_depth=5)`, `learning_rate=0.1`, `n_estimators=200`, `loss="linear"`. Bu sonuç iki şeyi doğruluyor: (1) `max_depth=3` olan varsayılan zayıf öğrenici bu problem için fazla basitmiş; (2) beklendiği gibi `"linear"` loss kazandı — uç fiyatlı lüks araçlar nedeniyle `"square"` ve `"exponential"` aşırı ceza uygulayıp modeli bozuyor.
* Düşük `learning_rate` (0.1) ile yüksek `n_estimators` (200) birlikte seçildi; bu, ikisi arasındaki **ters ilişkinin** somut bir örneği.

**Geliştirilebilecek yönler:** Hedef değişkenin sağa çarpık dağılımı nedeniyle `np.log1p(y)` dönüşümü denenmesi, uç fiyatlı lüks araçların ayrı ele alınması, ve AdaBoost yerine bu tür tablo verilerinde genellikle daha güçlü olan **Gradient Boosting / XGBoost** ile karşılaştırma yapılması.

**🇬🇧 English:**
Over the course of this notebook:

* Categorical columns were separated **by cardinality**: low-cardinality ones were encoded with `OneHotEncoder`, high-cardinality ones (`brand`, `model`) with `TargetEncoder`.
* `car_name` was recognized as **redundant** with `brand` + `model` and dropped.
* A concrete error taught that `ColumnTransformer` runs its transformers **in parallel**, so `StandardScaler` sees raw columns rather than the encoder's output; the scaler was chained inside the encoder's pipeline instead.
* `TargetEncoder` was found to infer the target type from the dtype of `y` — mistaking an integer price column for a classification target — and was pinned down with `target_type="continuous"`.
* **Data leakage** was avoided by fitting the preprocessing on the training set only.
* The default model scored **R² = 0.633**; tuning `n_estimators`, `learning_rate`, `loss`, and tree depth together with `GridSearchCV` lifted it to **R² = 0.907** (MSE 2.76×10¹¹ → 7.01×10¹⁰ — the error dropped to **a quarter** of its original size).
* The winning combination was `DecisionTreeRegressor(max_depth=5)`, `learning_rate=0.1`, `n_estimators=200`, `loss="linear"`. This confirms two things: (1) the default `max_depth=3` weak learner was too simple for this problem; (2) `"linear"` loss won as expected — `"square"` and `"exponential"` over-penalize the extreme-priced luxury cars and distort the model.
* A low `learning_rate` (0.1) was selected together with a high `n_estimators` (200) — a concrete illustration of the **inverse relationship** between the two.

**Possible improvements:** trying an `np.log1p(y)` transform given the right-skewed target distribution, handling the extreme-priced luxury cars separately, and benchmarking against **Gradient Boosting / XGBoost**, which usually outperforms AdaBoost on tabular data of this kind.